# 03 — Isolation Forest for Fraud Anomaly Detection

This notebook reconstructs the Isolation Forest experiment used in the project.

Methodology:
- Unsupervised training: `fraud` labels are **not** passed to the model
- Temporal train / validation / test split is reused from preprocessing
- Isolation Forest anomaly score is defined as `-decision_function(...)`
  so that **higher score = more anomalous**
- Thresholds are selected **only on validation**
- Final performance is measured on the untouched test set
- Model, scores, thresholds and metrics are saved at the end


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

RANDOM_STATE = 42


## Paths

In [2]:
PROCESSED_PATH = Path("../data/processed")
MODELS_PATH = Path("../models")
RESULTS_PATH = Path("../results")
SCORES_PATH = RESULTS_PATH / "scores"
METRICS_PATH = RESULTS_PATH / "metrics"

MODELS_PATH.mkdir(parents=True, exist_ok=True)
SCORES_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)


## Load preprocessed data

In [3]:
X_train = sparse.load_npz(
    PROCESSED_PATH / "X_train.npz"
)

X_valid = sparse.load_npz(
    PROCESSED_PATH / "X_valid.npz"
)

X_test = sparse.load_npz(
    PROCESSED_PATH / "X_test.npz"
)

y_train = np.load(
    PROCESSED_PATH / "y_train.npy"
)

y_valid = np.load(
    PROCESSED_PATH / "y_valid.npy"
)

y_test = np.load(
    PROCESSED_PATH / "y_test.npy"
)

print("Train:", X_train.shape, y_train.shape)
print("Valid:", X_valid.shape, y_valid.shape)
print("Test :", X_test.shape, y_test.shape)


Train: (374914, 79) (374914,)
Valid: (108423, 79) (108423,)
Test : (111306, 79) (111306,)


Expected shapes from preprocessing:

- Train: `(374914, 79)`
- Validation: `(108423, 79)`
- Test: `(111306, 79)`


## Build Isolation Forest

In [4]:
isolation_forest = IsolationForest(
    n_estimators=200,
    max_samples=4096,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

isolation_forest


IsolationForest(max_samples=4096, n_estimators=200, n_jobs=-1, random_state=42)

The detector is trained **without `y_train`**.

`fraud` labels are only used later for evaluation and validation-based threshold selection.


## Train

In [5]:
isolation_forest.fit(X_train)

print("Isolation Forest training complete.")


Isolation Forest training complete.


## Compute anomaly scores

Scikit-learn's `decision_function` uses the following convention:

- higher / positive → more normal
- lower / negative → more anomalous

For consistency with the rest of the project, we invert it:

`anomaly_score = -decision_function(X)`

Therefore:

- higher score → more suspicious
- lower score → more normal

These scores are **not probabilities**.


In [6]:
valid_scores = -isolation_forest.decision_function(
    X_valid
)

test_scores = -isolation_forest.decision_function(
    X_test
)

print("Validation scores:", valid_scores.shape)
print("Test scores      :", test_scores.shape)


Validation scores: (108423,)
Test scores      : (111306,)


## Validation score distribution

In [7]:
valid_score_analysis = pd.DataFrame({
    "fraud": y_valid,
    "anomaly_score": valid_scores
})

valid_score_analysis.groupby(
    "fraud"
)["anomaly_score"].describe()


,count,mean,std,min,25%,50%,75%,max
fraud,,,,,,,,
0,107223.0,-0.138520,0.023943,-0.158513,-0.153992,-0.147051,-0.136474,-0.010253
1,1200.0,-0.071261,0.015296,-0.108072,-0.081459,-0.072532,-0.061282,0.007999


Historical run reference:

- legitimate mean score ≈ `-0.1385`
- legitimate median ≈ `-0.1471`
- fraud mean score ≈ `-0.0713`
- fraud median ≈ `-0.0725`

Fraud transactions received higher anomaly scores on average.


## Ranking metrics

In [8]:
valid_roc_auc = roc_auc_score(
    y_valid,
    valid_scores
)

valid_pr_auc = average_precision_score(
    y_valid,
    valid_scores
)

test_roc_auc = roc_auc_score(
    y_test,
    test_scores
)

test_pr_auc = average_precision_score(
    y_test,
    test_scores
)

print(
    f"Validation ROC-AUC : "
    f"{valid_roc_auc:.4f}"
)

print(
    f"Validation PR-AUC  : "
    f"{valid_pr_auc:.4f}"
)

print()

print(
    f"Test ROC-AUC       : "
    f"{test_roc_auc:.4f}"
)

print(
    f"Test PR-AUC        : "
    f"{test_pr_auc:.4f}"
)


Validation ROC-AUC : 0.9654
Validation PR-AUC  : 0.2112

Test ROC-AUC       : 0.9672
Test PR-AUC        : 0.2102


Historical run reference:

- Validation ROC-AUC ≈ **0.9654**
- Validation PR-AUC ≈ **0.2112**
- Test ROC-AUC ≈ **0.9672**
- Test PR-AUC ≈ **0.2102**

The fraud prevalence is only around 1.2%, so a PR-AUC around 0.21 represents a large enrichment over random ranking.


## Validation threshold — Max F1

In [9]:
precision_if, recall_if, thresholds_if = (
    precision_recall_curve(
        y_valid,
        valid_scores
    )
)

# precision and recall have one extra element compared
# with thresholds. The final PR point has no threshold.
f1_scores_if = (
    2
    * precision_if[:-1]
    * recall_if[:-1]
    / (
        precision_if[:-1]
        + recall_if[:-1]
        + 1e-12
    )
)

best_index_if = np.argmax(
    f1_scores_if
)

best_threshold_if = thresholds_if[
    best_index_if
]

best_precision_if = precision_if[
    best_index_if
]

best_recall_if = recall_if[
    best_index_if
]

best_f1_if = f1_scores_if[
    best_index_if
]

print(
    f"Best threshold : "
    f"{best_threshold_if:.6f}"
)

print(
    f"Precision      : "
    f"{best_precision_if:.4f}"
)

print(
    f"Recall         : "
    f"{best_recall_if:.4f}"
)

print(
    f"F1-score       : "
    f"{best_f1_if:.4f}"
)


Best threshold : -0.070113
Precision      : 0.2105
Recall         : 0.4458
F1-score       : 0.2860


Historical validation reference:

- threshold ≈ `-0.070113`
- precision ≈ `0.2105`
- recall ≈ `0.4458`
- F1 ≈ `0.2860`


## Validation threshold — High Recall

In [10]:
TARGET_RECALL = 0.80

precision_thresholds_if = precision_if[:-1]
recall_thresholds_if = recall_if[:-1]

candidate_indices = np.where(
    recall_thresholds_if >= TARGET_RECALL
)[0]

best_recall_index_if = candidate_indices[
    np.argmax(
        precision_thresholds_if[
            candidate_indices
        ]
    )
]

recall80_threshold_if = thresholds_if[
    best_recall_index_if
]

recall80_precision_if = (
    precision_thresholds_if[
        best_recall_index_if
    ]
)

recall80_recall_if = (
    recall_thresholds_if[
        best_recall_index_if
    ]
)

recall80_f1_if = (
    2
    * recall80_precision_if
    * recall80_recall_if
    / (
        recall80_precision_if
        + recall80_recall_if
        + 1e-12
    )
)


In [11]:
if_validation_thresholds = pd.DataFrame({
    "Strategy": [
        "Max F1",
        "Recall >= 80%"
    ],
    "Threshold": [
        best_threshold_if,
        recall80_threshold_if
    ],
    "Precision": [
        best_precision_if,
        recall80_precision_if
    ],
    "Recall": [
        best_recall_if,
        recall80_recall_if
    ],
    "F1": [
        best_f1_if,
        recall80_f1_if
    ]
})

if_validation_thresholds


,Strategy,Threshold,Precision,Recall,F1
0,Max F1,-0.070113,0.210547,0.445833,0.286020
1,Recall >= 80%,-0.083048,0.136776,0.803333,0.233754


Historical validation reference for the high-recall strategy:

- threshold ≈ `-0.083048`
- precision ≈ `0.1368`
- recall ≈ `0.8033`
- F1 ≈ `0.2338`


## Apply frozen validation thresholds to test

In [12]:
y_test_pred_if_f1 = (
    test_scores >= best_threshold_if
).astype(int)

y_test_pred_if_r80 = (
    test_scores >= recall80_threshold_if
).astype(int)


## Evaluation helper

In [13]:
def evaluate_predictions(
    y_true,
    y_pred
):
    precision = precision_score(
        y_true,
        y_pred
    )

    recall = recall_score(
        y_true,
        y_pred
    )

    f1 = f1_score(
        y_true,
        y_pred
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    false_positive_rate = (
        fp / (fp + tn)
    )

    alert_rate = y_pred.mean()

    return {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "False Positive Rate": false_positive_rate,
        "Alert Rate": alert_rate,
        "True Positives": tp,
        "False Positives": fp,
        "False Negatives": fn,
        "True Negatives": tn
    }


## Final test evaluation

In [14]:
results_if_f1 = evaluate_predictions(
    y_test,
    y_test_pred_if_f1
)

results_if_r80 = evaluate_predictions(
    y_test,
    y_test_pred_if_r80
)


In [15]:
if_test_results = pd.DataFrame([
    {
        "Strategy": "Max F1",
        "Precision": results_if_f1["Precision"],
        "Recall": results_if_f1["Recall"],
        "F1": results_if_f1["F1"],
        "False Positive Rate":
            results_if_f1["False Positive Rate"],
        "Alert Rate":
            results_if_f1["Alert Rate"]
    },
    {
        "Strategy": "Recall >= 80%",
        "Precision": results_if_r80["Precision"],
        "Recall": results_if_r80["Recall"],
        "F1": results_if_r80["F1"],
        "False Positive Rate":
            results_if_r80["False Positive Rate"],
        "Alert Rate":
            results_if_r80["Alert Rate"]
    }
])

if_test_results


,Strategy,Precision,Recall,F1,False Positive Rate,Alert Rate
0,Max F1,0.211494,0.460000,0.289764,0.018691,0.023449
1,Recall >= 80%,0.133999,0.815833,0.230190,0.057463,0.065639


In [16]:
for strategy, results in [
    ("Max F1", results_if_f1),
    ("Recall >= 80%", results_if_r80)
]:
    print(f"\n--- {strategy} ---")
    print(
        "Frauds detected :",
        f"{results['True Positives']:,}"
    )
    print(
        "Frauds missed   :",
        f"{results['False Negatives']:,}"
    )
    print(
        "False alerts    :",
        f"{results['False Positives']:,}"
    )
    print(
        "True negatives  :",
        f"{results['True Negatives']:,}"
    )



--- Max F1 ---
Frauds detected : 552
Frauds missed   : 648
False alerts    : 2,058
True negatives  : 108,048

--- Recall >= 80% ---
Frauds detected : 979
Frauds missed   : 221
False alerts    : 6,327
True negatives  : 103,779


Historical test reference:

### Max F1
- Precision ≈ **0.2115**
- Recall ≈ **0.4600**
- F1 ≈ **0.2898**
- FPR ≈ **0.0187**
- Alert Rate ≈ **0.0234**

### High Recall
- Precision ≈ **0.1340**
- Recall ≈ **0.8158**
- F1 ≈ **0.2302**
- FPR ≈ **0.0575**
- Alert Rate ≈ **0.0656**

Use the values actually produced by your new run if you retrain.


## Save trained model

In [17]:
joblib.dump(
    isolation_forest,
    MODELS_PATH / "isolation_forest.joblib"
)

print("Isolation Forest saved.")


Isolation Forest saved.


## Save anomaly scores

In [18]:
np.save(
    SCORES_PATH / "iforest_valid_scores.npy",
    valid_scores
)

np.save(
    SCORES_PATH / "iforest_test_scores.npy",
    test_scores
)

print("Isolation Forest scores saved.")


Isolation Forest scores saved.


## Save thresholds

In [19]:
thresholds_to_save = {
    "max_f1": {
        "threshold":
            float(best_threshold_if),
        "validation_precision":
            float(best_precision_if),
        "validation_recall":
            float(best_recall_if),
        "validation_f1":
            float(best_f1_if)
    },

    "high_recall": {
        "target_recall":
            TARGET_RECALL,
        "threshold":
            float(recall80_threshold_if),
        "validation_precision":
            float(recall80_precision_if),
        "validation_recall":
            float(recall80_recall_if),
        "validation_f1":
            float(recall80_f1_if)
    }
}

with open(
    MODELS_PATH / "isolation_forest_thresholds.json",
    "w"
) as f:
    json.dump(
        thresholds_to_save,
        f,
        indent=2
    )

print("Thresholds saved.")


Thresholds saved.


## Save metrics

In [20]:
ranking_metrics_if = pd.DataFrame([
    {
        "Model": "Isolation Forest",
        "Validation ROC-AUC":
            valid_roc_auc,
        "Validation PR-AUC":
            valid_pr_auc,
        "Test ROC-AUC":
            test_roc_auc,
        "Test PR-AUC":
            test_pr_auc
    }
])

ranking_metrics_if.to_csv(
    METRICS_PATH
    / "isolation_forest_ranking_metrics.csv",
    index=False
)

if_test_results.to_csv(
    METRICS_PATH
    / "isolation_forest_operational_metrics.csv",
    index=False
)

print("Metrics saved.")


Metrics saved.


## Final verification

In [21]:
print("MODEL")
print(
    MODELS_PATH
    / "isolation_forest.joblib"
)

print("\nSCORES")
print(
    SCORES_PATH
    / "iforest_valid_scores.npy"
)
print(
    SCORES_PATH
    / "iforest_test_scores.npy"
)

print("\nTHRESHOLDS")
print(
    MODELS_PATH
    / "isolation_forest_thresholds.json"
)

print("\nMETRICS")
print(
    METRICS_PATH
    / "isolation_forest_ranking_metrics.csv"
)
print(
    METRICS_PATH
    / "isolation_forest_operational_metrics.csv"
)

print("\nDone.")


MODEL
..\models\isolation_forest.joblib

SCORES
..\results\scores\iforest_valid_scores.npy
..\results\scores\iforest_test_scores.npy

THRESHOLDS
..\models\isolation_forest_thresholds.json

METRICS
..\results\metrics\isolation_forest_ranking_metrics.csv
..\results\metrics\isolation_forest_operational_metrics.csv

Done.


## Conclusion

Isolation Forest provides a strong CPU-friendly anomaly-detection baseline.

Key findings from the original experiment:
- very strong ROC-AUC
- useful PR-AUC despite heavy class imbalance
- stable validation-to-test performance
- Max-F1 mode provides a selective operating point
- High-Recall mode captures substantially more fraud at the cost of more false alerts

The detector itself remains unsupervised: labels are used only for validation, threshold selection and final evaluation.
